# 08 - Prediction-Targeted Seg-Grad-CAM for Bidirectional ConvLSTM U-Net

This notebook generates prediction-targeted Seg-Grad-CAM explanations for the trained bidirectional ConvLSTM U-Net on the official EchoNet-Dynamic test split. It explains the model's own predicted foreground segmentation for each target ED/ES frame, not the ground-truth mask.

The notebook is designed for Kaggle with a GPU T4 x2 accelerator, but it intentionally uses one GPU (`cuda:0`) with batch size 1 for memory stability and correct activation-gradient association. The second T4 is left unused.

Do not hard-code local macOS paths in Kaggle. Attach the project code, EchoNet-Dynamic data, processed masks/metadata, and trained checkpoints as Kaggle datasets, then update the configuration cell below.


## Kaggle Inputs

Attach or upload datasets with these components:

- Project code containing the repository's `src` package or a flat folder with the same Python files.
- Raw EchoNet-Dynamic data containing `FileList.csv`, `VolumeTracings.csv`, and `Videos/*.avi`.
- Processed training artifacts containing `metadata.csv`, `images/`, and `masks/`.
- Trained bidirectional ConvLSTM U-Net checkpoints, preferably `best_model.pt` and `config.json` from the training run.

Outputs are written incrementally to `/kaggle/working/seg_gradcam_outputs` by default. Before ending the session, download this folder or create a Kaggle Dataset from it because `/kaggle/working` is not permanent after the session is discarded.


In [ ]:
# Optional Kaggle setup. Skip if the environment already has these packages.
%pip install -q monai opencv-python-headless pandas matplotlib tqdm


In [ ]:
from __future__ import annotations

import gc
import hashlib
import importlib
import json
import math
import os
import random
import shutil
import sys
import tempfile
import warnings
from contextlib import contextmanager
from pathlib import Path
from typing import Any

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader
from tqdm.auto import tqdm

# -----------------------------
# Kaggle path configuration
# -----------------------------
# Update these paths for your attached Kaggle datasets.
PROJECT_ROOT = Path("/kaggle/working/Echonet_temporal_XAI")
DATA_ROOT = Path("/kaggle/input")
RAW_DIR = Path("/kaggle/input/echonet-dynamic-raw/EchoNet-Dynamic")
PROCESSED_DIR = Path("/kaggle/input/echonet-dynamic-processed-masks")
CHECKPOINT_DIR = Path("/kaggle/input/bidirectional-convlstm-unet-07-11/checkpoints")
OUTPUT_ROOT = Path("/kaggle/working/seg_gradcam_outputs")

# If your Kaggle dataset contains files directly in a flat src-updated folder,
# set PROJECT_ROOT to that folder and FLAT_SOURCE_LAYOUT=True.
FLAT_SOURCE_LAYOUT = False

# Explanation and data configuration. These match the trained bidirectional model.
NUM_FRAMES_BEFORE = 12
NUM_FRAMES_AFTER = 12
TEMPORAL_STRIDE = 2
TARGET_IDX = NUM_FRAMES_BEFORE
SEQUENCE_LENGTH = NUM_FRAMES_BEFORE + 1 + NUM_FRAMES_AFTER
IMAGE_SIZE = (112, 112)
CHANNELS = (16, 32, 64, 128)
THRESHOLD = 0.5
SEED = 42

# Single-GPU Grad-CAM generation. Do not use DataParallel/DDP here.
DEVICE = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
BATCH_SIZE = 1
NUM_WORKERS = 0

# Run controls.
RUN_MODE = "smoke"  # "smoke" or "full"
SMOKE_MAX_SAMPLES = 2
FULL_MAX_SAMPLES = None
VISUALIZATION_TARGET_COUNT = 10
VISUALIZATION_SEED = 42
FAIL_FAST_SMOKE = True

# Motion-ablation controls. A few samples are additionally run with the target frame
# repeated across the full temporal sequence; normal full-sequence inference is preserved.
REPEATED_TARGET_ABLATION_ENABLED = True
REPEATED_TARGET_ABLATION_COUNT = 5  # keep between 5 and 10 for a lightweight motion check
REPEATED_TARGET_ABLATION_SEED = 42
REPEATED_TARGET_ABLATION_SUFFIX = "__target_repeat"

# Output layout.
ARRAY_DIR = OUTPUT_ROOT / "arrays"
VIS_DIR = OUTPUT_ROOT / "visualizations"
MANIFEST_DIR = OUTPUT_ROOT / "manifests"
METRICS_JSONL = MANIFEST_DIR / "sample_metrics.jsonl"
METRICS_CSV = MANIFEST_DIR / "sample_metrics.csv"
COMPLETION_MANIFEST = MANIFEST_DIR / "completed_samples.jsonl"
FAILURE_MANIFEST = MANIFEST_DIR / "failed_samples.jsonl"
SUMMARY_PATH = MANIFEST_DIR / "summary.json"
VISUALIZATION_MANIFEST = MANIFEST_DIR / "visualization_manifest.csv"

for directory in [OUTPUT_ROOT, ARRAY_DIR, VIS_DIR, MANIFEST_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print(f"PROJECT_ROOT: {PROJECT_ROOT}")
print(f"RAW_DIR: {RAW_DIR}")
print(f"PROCESSED_DIR: {PROCESSED_DIR}")
print(f"CHECKPOINT_DIR: {CHECKPOINT_DIR}")
print(f"OUTPUT_ROOT: {OUTPUT_ROOT}")
print(f"RUN_MODE: {RUN_MODE}")


In [ ]:
# Environment report for Kaggle T4 x2. We intentionally use only cuda:0.
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"Visible GPU count: {torch.cuda.device_count() if torch.cuda.is_available() else 0}")
if torch.cuda.is_available():
    for index in range(torch.cuda.device_count()):
        props = torch.cuda.get_device_properties(index)
        total_gb = props.total_memory / (1024 ** 3)
        free_text = ""
        try:
            free_bytes, total_bytes = torch.cuda.mem_get_info(index)
            free_text = f", free={free_bytes / (1024 ** 3):.2f} GiB"
        except Exception:
            pass
        print(f"GPU {index}: {props.name}, total={total_gb:.2f} GiB{free_text}")
    print(f"Selected GPU: {DEVICE} ({torch.cuda.get_device_name(DEVICE)})")
else:
    print("Selected device: CPU")


In [ ]:
# Import repository code. Supports either package layout PROJECT_ROOT/src/*.py or a flat source folder.
# The project files use relative imports, so flat folders are staged into a temporary src package.
def prepare_import_root(project_root: Path, flat_source_layout: bool) -> Path:
    if not flat_source_layout:
        if not (project_root / "src").exists():
            raise FileNotFoundError(
                f"Expected a src/ package under PROJECT_ROOT={project_root}. "
                "If your Kaggle dataset contains dataset.py and related files directly, set FLAT_SOURCE_LAYOUT=True."
            )
        return project_root

    staged_root = Path("/kaggle/working/_seg_gradcam_project_src") if Path("/kaggle/working").exists() else OUTPUT_ROOT / "_seg_gradcam_project_src"
    staged_src = staged_root / "src"
    staged_src.mkdir(parents=True, exist_ok=True)
    for py_file in project_root.glob("*.py"):
        shutil.copy2(py_file, staged_src / py_file.name)
    (staged_src / "__init__.py").touch()
    if not (staged_src / "dataset.py").exists():
        raise FileNotFoundError(f"Flat source folder did not contain dataset.py: {project_root}")
    return staged_root


IMPORT_ROOT = prepare_import_root(PROJECT_ROOT, FLAT_SOURCE_LAYOUT)
if str(IMPORT_ROOT) not in sys.path:
    sys.path.insert(0, str(IMPORT_ROOT))

model_module = importlib.import_module("src.bidirectional_convlstm_unet")
dataset_module = importlib.import_module("src.dataset")
train_module = importlib.import_module("src.temporal_train")
utils_module = importlib.import_module("src.utils")

build_bidirectional_convlstm_unet = model_module.build_bidirectional_convlstm_unet
EchoNetTemporalDataset = dataset_module.EchoNetTemporalDataset
load_temporal_metadata = dataset_module.load_temporal_metadata
split_by_echonet_filelist = dataset_module.split_by_echonet_filelist
get_temporal_loss = train_module.get_temporal_loss
segmentation_metrics = train_module.segmentation_metrics
load_echonet_tables = utils_module.load_echonet_tables
overlay_mask = utils_module.overlay_mask

print(f"Imported project modules successfully from: {IMPORT_ROOT}")


## Checkpoint Selection and Loading

The training code writes `best_model.pt` and `final_model.pt`. This notebook selects `best_model.pt` when present. If there is no reliable best checkpoint, it raises an error listing available files rather than silently choosing an arbitrary checkpoint.

Supported checkpoint formats:

- raw model state dictionary
- `checkpoint["model_state_dict"]`
- `checkpoint["state_dict"]`
- keys prefixed with `module.` from DataParallel-style training


In [ ]:
def list_checkpoint_files(checkpoint_dir: Path) -> list[Path]:
    if not checkpoint_dir.exists():
        raise FileNotFoundError(f"Checkpoint directory does not exist: {checkpoint_dir}")
    return sorted(
        path for path in checkpoint_dir.iterdir()
        if path.is_file() and path.suffix in {".pt", ".pth", ".ckpt"}
    )


def select_best_checkpoint(checkpoint_dir: Path) -> Path:
    files = list_checkpoint_files(checkpoint_dir)
    if not files:
        raise FileNotFoundError(f"No checkpoint files found in {checkpoint_dir}")

    exact_best = checkpoint_dir / "best_model.pt"
    if exact_best.exists():
        return exact_best

    best_like = [path for path in files if "best" in path.name.lower()]
    if len(best_like) == 1:
        return best_like[0]

    metric_candidates = []
    for path in files:
        try:
            checkpoint = torch.load(path, map_location="cpu")
        except Exception:
            continue
        if isinstance(checkpoint, dict):
            metrics = checkpoint.get("metrics", {})
            if isinstance(metrics, dict):
                dice = metrics.get("val_dice", metrics.get("dice"))
                if dice is not None:
                    metric_candidates.append((float(dice), path))
    if metric_candidates:
        metric_candidates.sort(key=lambda item: item[0], reverse=True)
        return metric_candidates[0][1]

    available = "\n".join(f"- {path.name}" for path in files)
    raise RuntimeError(
        "Could not reliably determine the best checkpoint. Expected best_model.pt, "
        "a unique filename containing 'best', or checkpoint metrics with val_dice/dice.\n"
        f"Available checkpoint files:\n{available}"
    )


def extract_state_dict(checkpoint: Any) -> dict[str, torch.Tensor]:
    if isinstance(checkpoint, dict):
        for key in ["model_state_dict", "state_dict"]:
            value = checkpoint.get(key)
            if isinstance(value, dict) and value:
                return value
        if checkpoint and all(isinstance(k, str) for k in checkpoint.keys()):
            tensor_values = [v for v in checkpoint.values() if isinstance(v, torch.Tensor)]
            if tensor_values:
                return checkpoint
    raise TypeError("Checkpoint does not contain a recognizable model state dictionary.")


def strip_module_prefix(state_dict: dict[str, torch.Tensor]) -> dict[str, torch.Tensor]:
    if not any(key.startswith("module.") for key in state_dict):
        return state_dict
    return {key.removeprefix("module."): value for key, value in state_dict.items()}


def checkpoint_config(checkpoint_path: Path) -> dict[str, Any]:
    candidates = [
        checkpoint_path.parent.parent / "config.json",
        checkpoint_path.parent / "config.json",
        CHECKPOINT_DIR.parent / "config.json",
    ]
    for path in candidates:
        if path.exists():
            with path.open("r", encoding="utf-8") as file:
                return json.load(file)
    return {}


CHECKPOINT_PATH = select_best_checkpoint(CHECKPOINT_DIR)
CHECKPOINT_ID = CHECKPOINT_PATH.stem
print(f"Selected checkpoint: {CHECKPOINT_PATH}")
ckpt_cfg = checkpoint_config(CHECKPOINT_PATH)
if ckpt_cfg:
    print("Checkpoint sidecar config:")
    print(json.dumps(ckpt_cfg, indent=2))
else:
    print("No config.json sidecar found; using notebook configuration.")


In [ ]:
def build_model_from_config() -> torch.nn.Module:
    channels = tuple(ckpt_cfg.get("channels", CHANNELS))
    num_frames_before = int(ckpt_cfg.get("num_frames_before", NUM_FRAMES_BEFORE))
    num_frames_after = int(ckpt_cfg.get("num_frames_after", NUM_FRAMES_AFTER))
    model = build_bidirectional_convlstm_unet(
        in_channels=1,
        out_channels=1,
        channels=channels,
        num_frames_before=num_frames_before,
        num_frames_after=num_frames_after,
    )
    return model.to(DEVICE)


def load_model_checkpoint(model: torch.nn.Module, checkpoint_path: Path) -> torch.nn.Module:
    checkpoint = torch.load(checkpoint_path, map_location=DEVICE)
    state_dict = strip_module_prefix(extract_state_dict(checkpoint))
    result = model.load_state_dict(state_dict, strict=False)
    missing = list(result.missing_keys)
    unexpected = list(result.unexpected_keys)
    print(f"Missing keys: {missing if missing else 'none'}")
    print(f"Unexpected keys: {unexpected if unexpected else 'none'}")
    if missing or unexpected:
        raise RuntimeError(
            "Checkpoint did not load cleanly. Review missing/unexpected keys above. "
            "This notebook requires all required model weights to load."
        )
    model.eval()
    return model


model = build_model_from_config()
model = load_model_checkpoint(model, CHECKPOINT_PATH)
loss_fn = get_temporal_loss()
print("Model loaded and set to eval mode.")


## Official Test Dataset

This uses the same repository dataset class as training: `EchoNetTemporalDataset`. It reconstructs official EchoNet-Dynamic train/validation/test partitions using `FileList.csv`, then keeps only the test samples. Frame sampling, boundary clamping, resizing, normalization, target indexing, and target-mask loading match the training dataset.


In [ ]:
def infer_phase_label(samples_for_video: list[dict[str, Any]], sample: dict[str, Any]) -> str:
    # EchoNet has two traced frames per video. Smaller frame index is usually ED and larger is ES in this processed table.
    same_video = sorted(int(item["frame_idx"]) for item in samples_for_video)
    frame_idx = int(sample["frame_idx"])
    if len(same_video) >= 2:
        if frame_idx == same_video[0]:
            return "ED"
        if frame_idx == same_video[-1]:
            return "ES"
    return "unknown"


def load_test_samples() -> list[dict[str, Any]]:
    metadata_path = PROCESSED_DIR / "metadata.csv"
    if not metadata_path.exists():
        raise FileNotFoundError(f"metadata.csv not found: {metadata_path}")
    if not (RAW_DIR / "FileList.csv").exists():
        raise FileNotFoundError(f"FileList.csv not found under RAW_DIR: {RAW_DIR}")
    if not (RAW_DIR / "Videos").exists():
        raise FileNotFoundError(f"Videos directory not found: {RAW_DIR / 'Videos'}")

    samples = load_temporal_metadata(metadata_path)
    file_list, _ = load_echonet_tables(RAW_DIR)
    _, _, test_samples = split_by_echonet_filelist(samples, file_list)
    by_video: dict[str, list[dict[str, Any]]] = {}
    for sample in samples:
        by_video.setdefault(str(sample["video_id"]), []).append(sample)
    enriched = []
    for sample in test_samples:
        item = dict(sample)
        item["phase"] = infer_phase_label(by_video.get(str(sample["video_id"]), []), item)
        enriched.append(item)
    return enriched


all_test_samples = load_test_samples()
active_samples = all_test_samples[:SMOKE_MAX_SAMPLES] if RUN_MODE == "smoke" else all_test_samples[:FULL_MAX_SAMPLES]
print(f"Total official test samples: {len(all_test_samples):,}")
print(f"Active samples for {RUN_MODE}: {len(active_samples):,}")
print(pd.Series([sample['phase'] for sample in active_samples]).value_counts(dropna=False))


test_dataset = EchoNetTemporalDataset(
    active_samples,
    videos_dir=RAW_DIR / "Videos",
    num_frames_before=NUM_FRAMES_BEFORE,
    num_frames_after=NUM_FRAMES_AFTER,
    temporal_stride=TEMPORAL_STRIDE,
    image_size=IMAGE_SIZE,
    augment=False,
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available(),
)

sample0 = test_dataset[0]
print(f"Sample sequence shape: {tuple(sample0['sequence'].shape)}")
print(f"Sample mask shape: {tuple(sample0['mask'].shape)}")
print(f"Target idx: {sample0['target_idx']}")
print(f"Frame indices: {sample0['frame_indices'].tolist()}")
assert sample0["sequence"].shape == (SEQUENCE_LENGTH, 1, *IMAGE_SIZE)
assert int(sample0["target_idx"]) == TARGET_IDX
assert int(sample0["frame_indices"][TARGET_IDX]) == int(sample0["frame_idx"])


## Prediction-Targeted Seg-Grad-CAM Capture

This notebook uses the Seg-Grad-CAM region-score idea from Vinogradova et al. for a binary target-frame segmentation model, adapting the official Keras implementation at https://github.com/kiraving/SegGradCAM to this PyTorch bidirectional ConvLSTM setting. The original Seg-Grad-CAM formulation sums class logits over a selected output-pixel region `M`. Here, `M` is the model's detached binary predicted-foreground mask at the target frame.

The target score is the mean pre-sigmoid foreground logit over `M`: sigmoid probabilities are used only to choose the predicted foreground region, and gradients flow through pre-sigmoid logits only. The threshold operation and the detached mask are not part of the differentiable gradient path, which avoids backpropagating through sigmoid saturation, a differentiable soft probability mask, thresholding, or the ground-truth mask. The mean is used instead of a sum to reduce foreground-area dependence when comparing raw CAM magnitude across frames and cardiac phases.

For each analyzed tensor, the CAM order is fixed as follows: calculate the channel-weighted signed CAM at the native feature-map resolution, upsample that signed CAM to the model input resolution using bilinear interpolation with `align_corners=False`, preserve the upsampled signed CAM, apply ReLU to obtain the raw positive CAM, then min-max normalize a copy only for visualization. Independently normalized CAMs are suitable for overlays but must not be used to compare absolute saliency magnitude across frames, because per-map normalization removes cross-frame scale information.

The analyzed representations are the shared encoder bottleneck for every input frame, forward ConvLSTM hidden states, and backward ConvLSTM hidden states. Encoder CAMs correspond directly to one input frame and are the primary representation for frame-level optical-flow comparisons. Forward and backward recurrent-state CAMs are cumulative memory explanations: a forward state summarizes information processed from the beginning of the sequence through that recurrent step, and a backward state summarizes information processed from the future side down to that recurrent step. Reversing the stored backward CAM order gives chronological indexing, but each backward CAM remains cumulative rather than an independent explanation of a single frame.


In [ ]:
class SegGradCAMCapture:
    def __init__(self, model: torch.nn.Module) -> None:
        self.model = model
        self.encoder_activations: list[torch.Tensor] = []
        self.forward_hidden_states: list[torch.Tensor] = []
        self.backward_hidden_states: list[torch.Tensor] = []
        self._encoder_handle = None
        self._original_run_temporal_branch = None

    def _encoder_hook(self, module, inputs, output):
        del module, inputs
        if not isinstance(output, torch.Tensor):
            raise TypeError("bottleneck_encoder output was not a tensor.")
        output.retain_grad()
        self.encoder_activations.append(output)

    def _patched_run_temporal_branch(self, cell, features, indices):
        state = None
        captured: list[torch.Tensor] | None = None
        if cell is self.model.forward_temporal_bottleneck:
            captured = self.forward_hidden_states
        elif cell is self.model.backward_temporal_bottleneck:
            captured = self.backward_hidden_states
        for time_idx in indices:
            encoded = features[time_idx]
            if state is None:
                state = cell.init_state(encoded)
            state = cell(encoded, state)
            hidden, _cell_state = state
            if captured is not None:
                hidden.retain_grad()
                captured.append(hidden)
        if state is None:
            raise RuntimeError("Temporal branch did not process any frames.")
        hidden, _ = state
        return hidden

    def __enter__(self):
        self._encoder_handle = self.model.bottleneck_encoder.register_forward_hook(self._encoder_hook)
        self._original_run_temporal_branch = self.model._run_temporal_branch
        self.model._run_temporal_branch = self._patched_run_temporal_branch
        return self

    def __exit__(self, exc_type, exc, tb):
        if self._encoder_handle is not None:
            self._encoder_handle.remove()
        if self._original_run_temporal_branch is not None:
            self.model._run_temporal_branch = self._original_run_temporal_branch
        return False


def prediction_target_score(logits: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor, int, bool]:
    probability = torch.sigmoid(logits)
    target_mask = (probability.detach() >= THRESHOLD).to(dtype=logits.dtype)
    empty_mask_fallback_used = bool(target_mask.sum().item() < 1.0)
    if empty_mask_fallback_used:
        flat_index = int(probability.detach().reshape(-1).argmax().item())
        target_mask = torch.zeros_like(logits)
        target_mask.reshape(-1)[flat_index] = 1.0
    target_mask = target_mask.detach()
    foreground_area = int(target_mask.sum().item())
    score = (logits * target_mask).sum() / target_mask.sum().clamp_min(1.0)
    return score, probability, target_mask, foreground_area, empty_mask_fallback_used


def normalize_positive_cam(positive_cam: torch.Tensor) -> tuple[torch.Tensor, bool]:
    cam_min = positive_cam.amin(dim=(2, 3), keepdim=True)
    cam_max = positive_cam.amax(dim=(2, 3), keepdim=True)
    normalized = (positive_cam - cam_min) / (cam_max - cam_min + 1e-8)
    after_zero = bool((normalized.detach().abs().amax() <= 1e-8).item())
    return normalized, after_zero


def cam_from_activation_gradient(activation: torch.Tensor, output_size: tuple[int, int]) -> dict[str, np.ndarray | bool]:
    gradient = activation.grad
    if gradient is None:
        raise RuntimeError("Required Grad-CAM tensor has gradient=None.")
    channel_weights = gradient.mean(dim=(2, 3), keepdim=True)
    signed_cam_native = (channel_weights * activation).sum(dim=1, keepdim=True)
    signed_cam = F.interpolate(signed_cam_native, size=output_size, mode="bilinear", align_corners=False)
    positive_cam = torch.relu(signed_cam)
    normalized_cam, zero_after_normalization = normalize_positive_cam(positive_cam)
    zero_before_normalization = bool((positive_cam.detach().abs().amax() <= 1e-8).item())
    return {
        "signed": signed_cam[0, 0].detach().cpu().numpy().astype(np.float32),
        "positive": positive_cam[0, 0].detach().cpu().numpy().astype(np.float32),
        "normalized": normalized_cam[0, 0].detach().cpu().numpy().astype(np.float32),
        "zero_before_normalization": zero_before_normalization,
        "zero_after_normalization": zero_after_normalization,
    }


def tensor_list_to_cam_package(
    tensors: list[torch.Tensor],
    output_size: tuple[int, int],
    representation: str,
    sample_id: str,
) -> tuple[dict[str, np.ndarray], list[dict[str, Any]]]:
    signed_cams, positive_cams, normalized_cams, zero_rows = [], [], [], []
    for position, tensor in enumerate(tensors):
        if tensor.grad is None:
            raise RuntimeError(
                f"Gradient was None for sample={sample_id}, representation={representation}, position={position}."
            )
        cam = cam_from_activation_gradient(tensor, output_size)
        signed_cams.append(cam["signed"])
        positive_cams.append(cam["positive"])
        normalized_cams.append(cam["normalized"])
        zero_rows.append({
            "representation": representation,
            "position": int(position),
            "zero_before_normalization": bool(cam["zero_before_normalization"]),
            "zero_after_normalization": bool(cam["zero_after_normalization"]),
        })
    return {
        "signed": np.stack(signed_cams, axis=0).astype(np.float32),
        "positive": np.stack(positive_cams, axis=0).astype(np.float32),
        "normalized": np.stack(normalized_cams, axis=0).astype(np.float32),
    }, zero_rows


def validate_target_score_outputs(score: torch.Tensor, target_mask: torch.Tensor, foreground_area: int) -> None:
    assert target_mask.requires_grad is False
    assert score.ndim == 0
    assert score.grad_fn is not None
    assert torch.isfinite(score).item()
    assert foreground_area >= 1


def validate_cam_package(package: dict[str, np.ndarray], expected_shape: tuple[int, int, int]) -> None:
    signed, positive, normalized = package["signed"], package["positive"], package["normalized"]
    assert signed.shape == expected_shape
    assert positive.shape == expected_shape
    assert normalized.shape == expected_shape
    assert signed.dtype == np.float32
    assert positive.dtype == np.float32
    assert normalized.dtype == np.float32
    assert np.nanmin(positive) >= -1e-7
    assert np.nanmin(normalized) >= -1e-6
    assert np.nanmax(normalized) <= 1.0 + 1e-6
    assert np.allclose(positive, np.maximum(signed, 0.0), atol=1e-5)


def run_single_seg_gradcam(model: torch.nn.Module, sequence: torch.Tensor, sample_id: str) -> dict[str, Any]:
    if sequence.shape[0] != 1:
        raise ValueError("Seg-Grad-CAM generation expects batch size 1.")
    model.eval()
    model.zero_grad(set_to_none=True)
    sequence = sequence.to(DEVICE, non_blocking=True).float()
    sequence.requires_grad_(False)
    output_size = tuple(sequence.shape[-2:])

    with SegGradCAMCapture(model) as capture:
        logits = model(sequence)
        assert logits.shape == (1, 1, *output_size), f"Unexpected output shape: {tuple(logits.shape)}"
        assert len(capture.encoder_activations) == SEQUENCE_LENGTH
        assert len(capture.forward_hidden_states) == TARGET_IDX + 1
        assert len(capture.backward_hidden_states) == NUM_FRAMES_AFTER + 1
        score, probability, target_mask, target_foreground_area, empty_mask_fallback_used = prediction_target_score(logits)
        validate_target_score_outputs(score, target_mask, target_foreground_area)
        score.backward()

        encoder_package, encoder_zero = tensor_list_to_cam_package(capture.encoder_activations, output_size, "encoder", sample_id)
        forward_package, forward_zero = tensor_list_to_cam_package(capture.forward_hidden_states, output_size, "forward", sample_id)
        backward_recurrent_package, backward_zero = tensor_list_to_cam_package(capture.backward_hidden_states, output_size, "backward_recurrent", sample_id)
        validate_cam_package(encoder_package, (SEQUENCE_LENGTH, *output_size))
        validate_cam_package(forward_package, (TARGET_IDX + 1, *output_size))
        validate_cam_package(backward_recurrent_package, (NUM_FRAMES_AFTER + 1, *output_size))
        backward_package = {key: value[::-1].copy().astype(np.float32) for key, value in backward_recurrent_package.items()}

    probability_np = probability[0, 0].detach().cpu().numpy().astype(np.float32)
    target_mask_np = target_mask[0, 0].detach().cpu().numpy().astype(np.uint8)
    segmentation_prediction = (probability_np >= THRESHOLD).astype(np.uint8)
    logits_cpu = logits.detach().cpu()
    result = {
        "logits": logits_cpu,
        "probability": probability_np,
        "prediction_mask": target_mask_np,
        "segmentation_prediction": segmentation_prediction,
        "target_score": float(score.detach().cpu().item()),
        "target_foreground_area": int(target_foreground_area),
        "empty_mask_fallback_used": bool(empty_mask_fallback_used),
        "encoder_signed_cam": encoder_package["signed"],
        "encoder_positive_cam": encoder_package["positive"],
        "encoder_normalized_cam": encoder_package["normalized"],
        "forward_signed_cam": forward_package["signed"],
        "forward_positive_cam": forward_package["positive"],
        "forward_normalized_cam": forward_package["normalized"],
        "backward_signed_cam": backward_package["signed"],
        "backward_positive_cam": backward_package["positive"],
        "backward_normalized_cam": backward_package["normalized"],
        "zero_rows": encoder_zero + forward_zero + backward_zero,
        "capture_counts": {
            "encoder_activations": len(capture.encoder_activations),
            "forward_hidden_states": len(capture.forward_hidden_states),
            "backward_hidden_states": len(capture.backward_hidden_states),
        },
    }
    del logits, probability, target_mask, score, sequence
    model.zero_grad(set_to_none=True)
    return result


## Output and Resume Utilities

Each completed sample writes one compressed `.npz` file atomically. If the file exists and passes a basic integrity check, the sample is skipped on resume. Metrics are also written incrementally to JSONL and later consolidated to CSV.


In [ ]:
def sanitize_id(value: str) -> str:
    keep = []
    for char in str(value):
        keep.append(char if char.isalnum() or char in {"-", "_"} else "_")
    return "".join(keep)


def sample_npz_path(sample_id: str) -> Path:
    safe = sanitize_id(sample_id)
    digest = hashlib.sha1(sample_id.encode("utf-8")).hexdigest()[:8]
    return ARRAY_DIR / f"{safe}_{digest}.npz"


def npz_integrity_ok(path: Path) -> bool:
    if not path.exists() or path.stat().st_size == 0:
        return False
    try:
        with np.load(path, allow_pickle=False) as data:
            required = [
                "probability", "prediction_mask", "segmentation_prediction",
                "encoder_signed_cam", "encoder_positive_cam", "encoder_normalized_cam",
                "forward_signed_cam", "forward_positive_cam", "forward_normalized_cam",
                "backward_signed_cam", "backward_positive_cam", "backward_normalized_cam",
                "target_score", "target_foreground_area", "predicted_foreground_area", "empty_mask_fallback_used",
                "frame_indices", "sequence_positions", "target_frame_idx",
                "zero_representation", "sample_id", "source_sample_id", "video_id", "phase", "input_condition",
                "repeated_target_frame_ablation", "checkpoint_id",
            ]
            if any(key not in data for key in required):
                return False
            # Touch every array so old object-dtype files fail here and are regenerated.
            for key in data.files:
                array = data[key]
                if array.dtype == object:
                    return False
            h, w = IMAGE_SIZE
            expected_shapes = {
                "encoder": (SEQUENCE_LENGTH, h, w),
                "forward": (TARGET_IDX + 1, h, w),
                "backward": (NUM_FRAMES_AFTER + 1, h, w),
            }
            checks = [
                data["target_foreground_area"].item() >= 1,
                data["predicted_foreground_area"].item() >= 1,
                np.array_equal(data["backward_chronological_order"], np.arange(TARGET_IDX, SEQUENCE_LENGTH, dtype=np.int64)),
                np.array_equal(data["forward_recurrent_step_indices"], np.arange(0, TARGET_IDX + 1, dtype=np.int64)),
            ]
            for prefix, expected_shape in expected_shapes.items():
                signed = data[f"{prefix}_signed_cam"]
                positive = data[f"{prefix}_positive_cam"]
                normalized = data[f"{prefix}_normalized_cam"]
                checks.extend([
                    signed.shape == expected_shape,
                    positive.shape == expected_shape,
                    normalized.shape == expected_shape,
                    signed.dtype == np.float32,
                    positive.dtype == np.float32,
                    normalized.dtype == np.float32,
                    np.nanmin(positive) >= -1e-7,
                    np.nanmin(normalized) >= -1e-6,
                    np.nanmax(normalized) <= 1.0 + 1e-6,
                    np.allclose(positive, np.maximum(signed, 0.0), atol=1e-5),
                ])
            return all(bool(check) for check in checks)
    except Exception:
        return False


def atomic_save_npz(path: Path, **arrays: Any) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with tempfile.NamedTemporaryFile(dir=path.parent, suffix=".npz", delete=False) as tmp:
        tmp_path = Path(tmp.name)
    try:
        np.savez_compressed(tmp_path, **arrays)
        tmp_path.replace(path)
    finally:
        if tmp_path.exists():
            tmp_path.unlink()


def append_jsonl(path: Path, row: dict[str, Any]) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("a", encoding="utf-8") as file:
        file.write(json.dumps(row) + "\n")


def read_jsonl(path: Path) -> pd.DataFrame:
    if not path.exists():
        return pd.DataFrame()
    rows = []
    with path.open("r", encoding="utf-8") as file:
        for raw_line in file:
            # Robust to older runs that wrote literal "\n" text instead of real JSONL newlines.
            for line in raw_line.replace("\\n", "\n").splitlines():
                line = line.strip()
                if line:
                    rows.append(json.loads(line))
    return pd.DataFrame(rows)


def batch_item(batch: dict[str, Any], key: str, index: int = 0):
    value = batch[key]
    if isinstance(value, torch.Tensor):
        selected = value[index]
        if selected.ndim == 0:
            return selected.detach().cpu().item()
        return selected.detach().cpu().numpy()
    if isinstance(value, (list, tuple)):
        return value[index]
    return value


def compute_scalar_metrics(logits: torch.Tensor, mask: torch.Tensor, loss_fn) -> tuple[float, float, float]:
    with torch.no_grad():
        loss = loss_fn(logits, mask).detach().cpu().item()
        dice_t, iou_t = segmentation_metrics(logits, mask)
    return float(dice_t.detach().cpu()[0]), float(iou_t.detach().cpu()[0]), float(loss)


def save_sample_arrays(
    path: Path,
    result: dict[str, Any],
    batch: dict[str, Any],
    sample_meta: dict[str, Any],
) -> None:
    frame_indices = np.asarray(batch_item(batch, "frame_indices"), dtype=np.int64)
    sequence_positions = np.arange(SEQUENCE_LENGTH, dtype=np.int64)
    forward_order = np.arange(0, TARGET_IDX + 1, dtype=np.int64)
    backward_recurrent_order = np.arange(SEQUENCE_LENGTH - 1, TARGET_IDX - 1, -1, dtype=np.int64)
    backward_chronological_order = backward_recurrent_order[::-1].copy()
    zero_table = pd.DataFrame(result["zero_rows"])
    ground_truth_mask = batch_item(batch, "mask")[0].astype(np.float32)
    atomic_save_npz(
        path,
        probability=result["probability"].astype(np.float32),
        prediction_mask=result["prediction_mask"].astype(np.uint8),
        segmentation_prediction=result["segmentation_prediction"].astype(np.uint8),
        ground_truth_mask=ground_truth_mask,
        encoder_signed_cam=result["encoder_signed_cam"].astype(np.float32),
        encoder_positive_cam=result["encoder_positive_cam"].astype(np.float32),
        encoder_normalized_cam=result["encoder_normalized_cam"].astype(np.float32),
        forward_signed_cam=result["forward_signed_cam"].astype(np.float32),
        forward_positive_cam=result["forward_positive_cam"].astype(np.float32),
        forward_normalized_cam=result["forward_normalized_cam"].astype(np.float32),
        backward_signed_cam=result["backward_signed_cam"].astype(np.float32),
        backward_positive_cam=result["backward_positive_cam"].astype(np.float32),
        backward_normalized_cam=result["backward_normalized_cam"].astype(np.float32),
        target_score=np.asarray(result["target_score"], dtype=np.float32),
        target_foreground_area=np.asarray(result["target_foreground_area"], dtype=np.int64),
        predicted_foreground_area=np.asarray(result["target_foreground_area"], dtype=np.int64),
        empty_mask_fallback_used=np.asarray(result["empty_mask_fallback_used"], dtype=bool),
        frame_indices=frame_indices,
        sequence_positions=sequence_positions,
        target_idx=np.asarray(TARGET_IDX, dtype=np.int64),
        target_frame_idx=np.asarray(int(sample_meta["target_frame_idx"]), dtype=np.int64),
        forward_recurrent_order=forward_order,
        forward_recurrent_step_indices=forward_order,
        backward_recurrent_order=backward_recurrent_order,
        backward_chronological_order=backward_chronological_order,
        backward_recurrent_step_indices_chronological=backward_chronological_order,
        analyzed_layer_names=np.asarray(["encoder_bottleneck", "forward_convlstm_hidden", "backward_convlstm_hidden"], dtype="U64"),
        zero_representation=zero_table["representation"].astype(str).to_numpy(dtype="U32"),
        zero_position=zero_table["position"].to_numpy(dtype=np.int64),
        zero_before_normalization=zero_table["zero_before_normalization"].to_numpy(dtype=bool),
        zero_after_normalization=zero_table["zero_after_normalization"].to_numpy(dtype=bool),
        sample_id=np.asarray(str(sample_meta["sample_id"]), dtype="U128"),
        source_sample_id=np.asarray(str(sample_meta.get("source_sample_id", sample_meta["sample_id"])), dtype="U128"),
        video_id=np.asarray(str(sample_meta["video_id"]), dtype="U128"),
        phase=np.asarray(str(sample_meta["phase"]), dtype="U16"),
        input_condition=np.asarray(str(sample_meta.get("input_condition", "normal")), dtype="U32"),
        repeated_target_frame_ablation=np.asarray(bool(sample_meta.get("repeated_target_frame_ablation", False)), dtype=bool),
        checkpoint_id=np.asarray(str(CHECKPOINT_ID), dtype="U128"),
    )


## Smoke Test

The optional repeated-target-frame ablation tests whether the trained temporal model changes when motion information is removed. For selected samples, the notebook creates an additional inference condition where the original target frame is copied across every temporal position, while all other samples still use the normal dataloader sequence. These ablation rows use sample IDs ending in `__target_repeat` and are saved separately from the normal full-sequence outputs.


In [ ]:
def selected_repeated_target_ablation_indices(total_samples: int) -> set[int]:
    if not REPEATED_TARGET_ABLATION_ENABLED or total_samples <= 0:
        return set()
    count = min(max(int(REPEATED_TARGET_ABLATION_COUNT), 0), 10, total_samples)
    if count == 0:
        return set()
    rng = np.random.default_rng(REPEATED_TARGET_ABLATION_SEED)
    return set(int(idx) for idx in rng.choice(np.arange(total_samples), size=count, replace=False))


REPEATED_TARGET_ABLATION_INDICES = selected_repeated_target_ablation_indices(len(active_samples))
print(f"Repeated-target ablation samples: {len(REPEATED_TARGET_ABLATION_INDICES)}")
if REPEATED_TARGET_ABLATION_INDICES:
    print("Ablation dataset indices:", sorted(REPEATED_TARGET_ABLATION_INDICES))


def repeated_target_batch(batch: dict[str, Any]) -> dict[str, Any]:
    ablated = dict(batch)
    sequence = batch["sequence"].clone()
    target_frames = sequence[:, TARGET_IDX:TARGET_IDX + 1].clone()
    ablated["sequence"] = target_frames.repeat(1, SEQUENCE_LENGTH, 1, 1, 1).contiguous()
    if "frame_indices" in batch and isinstance(batch["frame_indices"], torch.Tensor):
        target_frame_indices = batch["frame_indices"][:, TARGET_IDX:TARGET_IDX + 1].clone()
        ablated["frame_indices"] = target_frame_indices.repeat(1, SEQUENCE_LENGTH).contiguous()
    return ablated


def sample_metadata_for_index(index: int, batch: dict[str, Any], input_condition: str = "normal") -> dict[str, Any]:
    sample = active_samples[index]
    source_sample_id = str(batch_item(batch, "id"))
    sample_id = source_sample_id
    if input_condition == "target_repeat":
        sample_id = f"{source_sample_id}{REPEATED_TARGET_ABLATION_SUFFIX}"
    return {
        "sample_id": sample_id,
        "source_sample_id": source_sample_id,
        "video_id": str(batch_item(batch, "video_id")),
        "phase": str(sample.get("phase", "unknown")),
        "target_frame_idx": int(batch_item(batch, "frame_idx")),
        "input_condition": input_condition,
        "repeated_target_frame_ablation": bool(input_condition == "target_repeat"),
    }


def process_one_batch(
    batch: dict[str, Any],
    dataset_index: int,
    smoke: bool = False,
    input_condition: str = "normal",
) -> dict[str, Any]:
    batch_for_inference = repeated_target_batch(batch) if input_condition == "target_repeat" else batch
    sequence = batch_for_inference["sequence"].to(DEVICE, non_blocking=True)
    mask = batch_for_inference["mask"].to(DEVICE, non_blocking=True)
    sample_meta = sample_metadata_for_index(dataset_index, batch, input_condition=input_condition)
    sample_id = sample_meta["sample_id"]
    output_path = sample_npz_path(sample_id)

    if npz_integrity_ok(output_path):
        return {"status": "skipped", "sample_id": sample_id, "input_condition": input_condition, "output_path": str(output_path)}

    result = run_single_seg_gradcam(model, sequence, sample_id)
    dice, iou, loss = compute_scalar_metrics(result["logits"].to(DEVICE), mask, loss_fn)
    pred_area = int(result["prediction_mask"].sum())
    target_score = float(result["target_score"])
    empty_mask_fallback_used = bool(result["empty_mask_fallback_used"])
    gt_area = int((mask.detach().cpu().numpy()[0, 0] > 0.5).sum())
    any_zero = any(row["zero_after_normalization"] for row in result["zero_rows"])

    if smoke:
        assert result["capture_counts"] == {
            "encoder_activations": SEQUENCE_LENGTH,
            "forward_hidden_states": TARGET_IDX + 1,
            "backward_hidden_states": NUM_FRAMES_AFTER + 1,
        }
        assert result["encoder_signed_cam"].shape == (SEQUENCE_LENGTH, *IMAGE_SIZE)
        assert result["encoder_positive_cam"].shape == (SEQUENCE_LENGTH, *IMAGE_SIZE)
        assert result["encoder_normalized_cam"].shape == (SEQUENCE_LENGTH, *IMAGE_SIZE)
        assert result["forward_signed_cam"].shape == (TARGET_IDX + 1, *IMAGE_SIZE)
        assert result["forward_positive_cam"].shape == (TARGET_IDX + 1, *IMAGE_SIZE)
        assert result["forward_normalized_cam"].shape == (TARGET_IDX + 1, *IMAGE_SIZE)
        assert result["backward_signed_cam"].shape == (NUM_FRAMES_AFTER + 1, *IMAGE_SIZE)
        assert result["backward_positive_cam"].shape == (NUM_FRAMES_AFTER + 1, *IMAGE_SIZE)
        assert result["backward_normalized_cam"].shape == (NUM_FRAMES_AFTER + 1, *IMAGE_SIZE)
        if input_condition == "target_repeat":
            repeated_indices = batch_item(batch_for_inference, "frame_indices")
            assert len(set(int(x) for x in repeated_indices)) == 1

    save_sample_arrays(output_path, result, batch_for_inference, sample_meta)
    rel_path = str(output_path.relative_to(OUTPUT_ROOT))
    row = {
        **sample_meta,
        "target_position": TARGET_IDX,
        "sampled_frame_indices": " ".join(str(int(x)) for x in batch_item(batch_for_inference, "frame_indices")),
        "checkpoint_id": CHECKPOINT_ID,
        "predicted_foreground_area": pred_area,
        "ground_truth_foreground_area": gt_area,
        "target_score": target_score,
        "empty_mask_fallback_used": empty_mask_fallback_used,
        "dice": dice,
        "iou": iou,
        "segmentation_loss": loss,
        "any_cam_zero": bool(any_zero),
        "relative_npz_path": rel_path,
    }
    append_jsonl(METRICS_JSONL, row)
    append_jsonl(COMPLETION_MANIFEST, {
        "sample_id": sample_id,
        "source_sample_id": sample_meta["source_sample_id"],
        "input_condition": input_condition,
        "relative_npz_path": rel_path,
    })

    del sequence, mask, result
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return {"status": "processed", **row}


if RUN_MODE == "smoke":
    smoke_rows = []
    for dataset_index, batch in enumerate(tqdm(test_loader, desc="Smoke Seg-Grad-CAM")):
        try:
            smoke_rows.append(process_one_batch(batch, dataset_index, smoke=True, input_condition="normal"))
            if dataset_index in REPEATED_TARGET_ABLATION_INDICES:
                smoke_rows.append(process_one_batch(batch, dataset_index, smoke=True, input_condition="target_repeat"))
        except Exception as exc:
            append_jsonl(FAILURE_MANIFEST, {"dataset_index": dataset_index, "error": repr(exc)})
            if FAIL_FAST_SMOKE:
                raise
    display(pd.DataFrame(smoke_rows))


## Full Test-Set Processing

In full mode, every active sample is processed with the normal temporal input sequence. A small selected subset is then processed a second time under the `target_repeat` ablation condition, where the target frame is repeated across all sequence positions. These ablation outputs are separate rows/files, so normal full-sequence inference remains unchanged.


In [ ]:
if RUN_MODE == "full":
    processed = skipped = failed = 0
    for dataset_index, batch in enumerate(tqdm(test_loader, desc="Full Seg-Grad-CAM")):
        planned_conditions = ["normal"]
        if dataset_index in REPEATED_TARGET_ABLATION_INDICES:
            planned_conditions.append("target_repeat")
        for input_condition in planned_conditions:
            try:
                result_row = process_one_batch(batch, dataset_index, smoke=False, input_condition=input_condition)
                if result_row["status"] == "skipped":
                    skipped += 1
                else:
                    processed += 1
            except Exception as exc:
                failed += 1
                sample_id = "unknown"
                try:
                    sample_id = str(batch_item(batch, "id"))
                    if input_condition == "target_repeat":
                        sample_id = f"{sample_id}{REPEATED_TARGET_ABLATION_SUFFIX}"
                except Exception:
                    pass
                append_jsonl(FAILURE_MANIFEST, {
                    "dataset_index": dataset_index,
                    "sample_id": sample_id,
                    "input_condition": input_condition,
                    "error": repr(exc),
                })
                print(f"Failed sample {dataset_index} / {sample_id} / {input_condition}: {exc}")
            finally:
                gc.collect()
                if torch.cuda.is_available():
                    torch.cuda.empty_cache()
    print(f"Full processing complete. processed={processed}, skipped={skipped}, failed={failed}")


## Consolidate Metrics

This cell rewrites a compact CSV from the incremental JSONL metrics file. It also drops duplicate sample IDs, keeping the latest row, which makes resume runs tidy.


In [ ]:
metrics_df = read_jsonl(METRICS_JSONL)
if metrics_df.empty:
    print("No metrics available yet.")
else:
    metrics_df = metrics_df.drop_duplicates(subset=["sample_id"], keep="last").reset_index(drop=True)
    metrics_df.to_csv(METRICS_CSV, index=False)
    print(f"Saved consolidated metrics: {METRICS_CSV}")
    display(metrics_df.head())
    display(metrics_df[["dice", "iou", "segmentation_loss", "predicted_foreground_area", "ground_truth_foreground_area"]].describe())

    if "input_condition" in metrics_df.columns and "source_sample_id" in metrics_df.columns:
        paired = metrics_df[metrics_df["input_condition"].isin(["normal", "target_repeat"])].copy()
        if not paired.empty:
            paired_summary = paired.pivot_table(
                index="source_sample_id",
                columns="input_condition",
                values=["dice", "iou", "segmentation_loss", "target_score", "predicted_foreground_area"],
                aggfunc="last",
            )
            if "normal" in paired_summary.columns.get_level_values(1) and "target_repeat" in paired_summary.columns.get_level_values(1):
                paired_summary[("delta", "dice_target_repeat_minus_normal")] = paired_summary[("dice", "target_repeat")] - paired_summary[("dice", "normal")]
                paired_summary[("delta", "iou_target_repeat_minus_normal")] = paired_summary[("iou", "target_repeat")] - paired_summary[("iou", "normal")]
                print("Normal vs repeated-target-frame ablation metrics:")
                display(paired_summary.dropna(how="all"))


## Select Visualization Subset

Visualization PNGs are intentionally limited. By default, the notebook writes 10 visualization sets: the two lowest-Dice samples, the two highest-Dice samples, the remaining slots selected randomly from other completed samples, and at least one `target_repeat` ablation sample when such rows are available.


In [ ]:
def select_visualization_subset(metrics_df: pd.DataFrame, target_count: int = 10, seed: int = 42) -> pd.DataFrame:
    if metrics_df.empty or target_count <= 0:
        return metrics_df.iloc[0:0].copy()

    metrics_df = metrics_df.copy().reset_index(drop=True)
    if "input_condition" not in metrics_df.columns:
        metrics_df["input_condition"] = "normal"
    if "repeated_target_frame_ablation" not in metrics_df.columns:
        metrics_df["repeated_target_frame_ablation"] = metrics_df["input_condition"].eq("target_repeat")
    target_count = min(int(target_count), len(metrics_df))
    selected_indices: list[int] = []

    def add_unique(indices: list[int]) -> None:
        for index in indices:
            index = int(index)
            if index not in selected_indices and len(selected_indices) < target_count:
                selected_indices.append(index)

    ablation_df = metrics_df[metrics_df["input_condition"].fillna("normal") == "target_repeat"]
    if not ablation_df.empty:
        add_unique(ablation_df.sort_values("dice", ascending=True, kind="mergesort").head(1).index.tolist())

    ranked = metrics_df.sort_values("dice", ascending=True, kind="mergesort")
    add_unique(ranked.head(2).index.tolist())
    add_unique(ranked.tail(2).sort_values("dice", ascending=False, kind="mergesort").index.tolist())

    remaining_slots = target_count - len(selected_indices)
    if remaining_slots > 0:
        remaining = [idx for idx in metrics_df.index.tolist() if idx not in selected_indices]
        if remaining:
            rng = np.random.default_rng(seed)
            random_indices = rng.choice(remaining, size=min(remaining_slots, len(remaining)), replace=False)
            add_unique([int(idx) for idx in random_indices])

    selected = metrics_df.loc[selected_indices].copy()
    selected["visualization_selection_reason"] = "random"
    low_ids = set(ranked.head(2).index.tolist())
    high_ids = set(ranked.tail(2).index.tolist())
    ablation_ids = set(ablation_df.index.tolist())
    selected.loc[selected.index.isin(low_ids), "visualization_selection_reason"] = "lowest_dice"
    selected.loc[selected.index.isin(high_ids), "visualization_selection_reason"] = "highest_dice"
    selected.loc[selected.index.isin(ablation_ids), "visualization_selection_reason"] = "target_repeat_ablation"
    return selected.reset_index(drop=True)


if not metrics_df.empty:
    vis_df = select_visualization_subset(metrics_df, VISUALIZATION_TARGET_COUNT, VISUALIZATION_SEED)
    vis_df.to_csv(VISUALIZATION_MANIFEST, index=False)
    print(f"Selected visualization samples: {len(vis_df):,}")
    display_columns = ["sample_id", "phase", "input_condition", "dice", "iou", "any_cam_zero", "visualization_selection_reason", "relative_npz_path"]
    display(vis_df[[column for column in display_columns if column in vis_df.columns]])
else:
    vis_df = pd.DataFrame()


## Visualization Writers

These cells load compact arrays already written by the full processing stage. They do not rerun Grad-CAM. Contact sheets keep the PNG count small while still showing all frames and all representation families.


In [ ]:
def load_sample_npz(relative_path: str) -> dict[str, Any]:
    path = OUTPUT_ROOT / relative_path
    with np.load(path, allow_pickle=False) as data:
        return {key: data[key] for key in data.files}


def read_sequence_and_mask_for_sample(
    sample_id: str,
    source_sample_id: str | None = None,
    input_condition: str = "normal",
) -> tuple[np.ndarray, np.ndarray, dict[str, Any]]:
    lookup_id = source_sample_id or sample_id.removesuffix(REPEATED_TARGET_ABLATION_SUFFIX)
    sample_lookup = {str(item["id"]): item for item in active_samples}
    sample = sample_lookup.get(lookup_id)
    if sample is None:
        # Fall back to all_test_samples so visualization can run after a full manifest reload.
        sample = {str(item["id"]): item for item in all_test_samples}.get(lookup_id)
    if sample is None:
        raise KeyError(f"Could not find sample metadata for {sample_id}; source={lookup_id}")
    ds = EchoNetTemporalDataset(
        [sample],
        videos_dir=RAW_DIR / "Videos",
        num_frames_before=NUM_FRAMES_BEFORE,
        num_frames_after=NUM_FRAMES_AFTER,
        temporal_stride=TEMPORAL_STRIDE,
        image_size=IMAGE_SIZE,
        augment=False,
    )
    item = ds[0]
    sequence = item["sequence"][:, 0].numpy()
    if input_condition == "target_repeat":
        target_frame = sequence[TARGET_IDX].copy()
        sequence = np.repeat(target_frame[None, ...], SEQUENCE_LENGTH, axis=0)
    mask = item["mask"][0].numpy()
    return sequence, mask, sample


def draw_target_outline(axis, shape: tuple[int, int], color: str = "cyan", linewidth: float = 3.0) -> None:
    height, width = int(shape[0]), int(shape[1])
    rectangle = plt.Rectangle(
        (0, 0),
        width - 1,
        height - 1,
        fill=False,
        edgecolor=color,
        linewidth=linewidth,
    )
    axis.add_patch(rectangle)


def make_contact_sheet(
    arrays: np.ndarray,
    titles: list[str] | None,
    path: Path,
    cmap: str = "gray",
    highlight_indices: set[int] | None = None,
    figure_title: str | None = None,
) -> None:
    n = arrays.shape[0]
    cols = min(5, n)
    rows = int(math.ceil(n / cols))
    fig, axes = plt.subplots(rows, cols, figsize=(cols * 2.2, rows * 2.2), squeeze=False)
    highlight_indices = highlight_indices or set()
    for idx, axis in enumerate(axes.flat):
        axis.axis("off")
        if idx < n:
            axis.imshow(arrays[idx], cmap=cmap, vmin=0, vmax=1)
            if idx in highlight_indices:
                draw_target_outline(axis, arrays[idx].shape, color="cyan", linewidth=3.0)
            if titles:
                axis.set_title(titles[idx], fontsize=8)
    if figure_title:
        fig.suptitle(figure_title, fontsize=11)
        fig.tight_layout(rect=(0, 0, 1, 0.96))
    else:
        fig.tight_layout()
    path.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(path, dpi=150, bbox_inches="tight")
    plt.close(fig)


def overlay_heatmap_on_frame(frame: np.ndarray, heatmap: np.ndarray, alpha: float = 0.45) -> np.ndarray:
    frame_uint8 = np.clip(frame * 255.0, 0, 255).astype(np.uint8)
    heat_uint8 = np.clip(heatmap * 255.0, 0, 255).astype(np.uint8)
    colored = cv2.applyColorMap(heat_uint8, cv2.COLORMAP_JET)
    frame_rgb = cv2.cvtColor(frame_uint8, cv2.COLOR_GRAY2BGR)
    blended = cv2.addWeighted(frame_rgb, 1.0 - alpha, colored, alpha, 0.0)
    return cv2.cvtColor(blended, cv2.COLOR_BGR2RGB)


def add_mask_footer(
    axes_row: np.ndarray,
    target_frame: np.ndarray,
    gt_mask: np.ndarray,
    pred_mask: np.ndarray,
) -> None:
    for axis in axes_row:
        axis.axis("off")
    panels = [
        (target_frame, "Target frame", "gray"),
        (gt_mask, "Ground truth mask", "gray"),
        (pred_mask.astype(np.float32), "Predicted mask", "gray"),
    ]
    for axis, (image, title, cmap) in zip(axes_row[:3], panels):
        axis.imshow(image, cmap=cmap, vmin=0, vmax=1)
        axis.set_title(title, fontsize=9)
        if title == "Target frame":
            draw_target_outline(axis, image.shape, color="cyan", linewidth=3.0)
    if len(axes_row) >= 4:
        axes_row[3].imshow(target_frame, cmap="gray", vmin=0, vmax=1)
        axes_row[3].contour(gt_mask > 0.5, colors="lime", linewidths=1.0)
        axes_row[3].contour(pred_mask > 0.5, colors="red", linewidths=1.0)
        axes_row[3].set_title("GT green / pred red", fontsize=9)
        axes_row[3].axis("off")


def overlay_sheet(
    frames: np.ndarray,
    heatmaps: np.ndarray,
    titles: list[str],
    path: Path,
    target_position: int | None = None,
    gt_mask: np.ndarray | None = None,
    pred_mask: np.ndarray | None = None,
    figure_title: str | None = None,
) -> None:
    overlays = np.stack([
        overlay_heatmap_on_frame(frames[min(i, len(frames) - 1)], heatmaps[i])
        for i in range(len(heatmaps))
    ])
    n = overlays.shape[0]
    cols = min(5, n)
    overlay_rows = int(math.ceil(n / cols))
    footer_rows = 1 if gt_mask is not None and pred_mask is not None else 0
    rows = overlay_rows + footer_rows
    fig, axes = plt.subplots(rows, cols, figsize=(cols * 2.4, rows * 2.4), squeeze=False)
    for idx in range(overlay_rows * cols):
        axis = axes.flat[idx]
        axis.axis("off")
        if idx < n:
            axis.imshow(overlays[idx])
            if target_position is not None and idx == target_position:
                draw_target_outline(axis, overlays[idx].shape[:2], color="cyan", linewidth=4.0)
            axis.set_title(titles[idx], fontsize=8)
    if footer_rows:
        add_mask_footer(axes[-1], frames[target_position if target_position is not None else TARGET_IDX], gt_mask, pred_mask)
    if figure_title:
        fig.suptitle(figure_title, fontsize=11)
        fig.tight_layout(rect=(0, 0, 1, 0.96))
    else:
        fig.tight_layout()
    path.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(path, dpi=150, bbox_inches="tight")
    plt.close(fig)


def combined_forward_backward_temporal_cams(arrays: dict[str, Any]) -> tuple[np.ndarray, list[str]]:
    forward = arrays["forward_normalized_cam"].astype(np.float32)
    backward = arrays["backward_normalized_cam"].astype(np.float32)
    combined = np.zeros((SEQUENCE_LENGTH, *forward.shape[-2:]), dtype=np.float32)
    combined[:TARGET_IDX] = forward[:TARGET_IDX]
    # Target frame has both target-aligned forward and backward hidden states; use max to preserve either branch's saliency.
    combined[TARGET_IDX] = np.maximum(forward[TARGET_IDX], backward[0])
    combined[TARGET_IDX + 1:] = backward[1:]
    titles = []
    for pos in range(SEQUENCE_LENGTH):
        if pos < TARGET_IDX:
            titles.append(f"pos {pos}\nF state {pos}")
        elif pos == TARGET_IDX:
            titles.append(f"pos {pos}\nF+B target")
        else:
            titles.append(f"pos {pos}\nB chron {pos}")
    return combined, titles


def contour_overlay_figure(row: pd.Series, sequence: np.ndarray, gt_mask: np.ndarray, arrays: dict[str, Any], output_dir: Path) -> None:
    target = sequence[TARGET_IDX]
    pred_mask = arrays["prediction_mask"].astype(np.uint8)
    target_cam = arrays["encoder_normalized_cam"][TARGET_IDX].astype(np.float32)
    fig, axes = plt.subplots(1, 5, figsize=(16, 4))
    axes[0].imshow(target, cmap="gray", vmin=0, vmax=1)
    draw_target_outline(axes[0], target.shape, color="cyan", linewidth=3.0)
    axes[0].set_title("Target frame")
    axes[1].imshow(gt_mask, cmap="gray", vmin=0, vmax=1)
    axes[1].set_title("Ground truth")
    axes[2].imshow(pred_mask, cmap="gray", vmin=0, vmax=1)
    axes[2].set_title("Prediction")
    axes[3].imshow(overlay_heatmap_on_frame(target, target_cam))
    draw_target_outline(axes[3], target.shape, color="cyan", linewidth=3.0)
    axes[3].set_title("Pred-target CAM")
    axes[4].imshow(target, cmap="gray", vmin=0, vmax=1)
    axes[4].contour(gt_mask > 0.5, colors="lime", linewidths=1.2)
    axes[4].contour(pred_mask > 0.5, colors="red", linewidths=1.2)
    axes[4].set_title("GT green / pred red")
    for axis in axes:
        axis.axis("off")
    fig.suptitle(
        f"{row['sample_id']} | {row['phase']} | Dice={row['dice']:.3f} | IoU={row['iou']:.3f}",
        fontsize=11,
    )
    fig.tight_layout()
    fig.savefig(output_dir / "target_comparison.png", dpi=150, bbox_inches="tight")
    plt.close(fig)


def write_visualization_set(row: pd.Series) -> dict[str, Any]:
    sample_id = str(row["sample_id"])
    output_dir = VIS_DIR / sanitize_id(sample_id)
    output_dir.mkdir(parents=True, exist_ok=True)
    arrays = load_sample_npz(str(row["relative_npz_path"]))
    input_condition_value = row.get("input_condition", "normal")
    input_condition = "normal" if pd.isna(input_condition_value) else str(input_condition_value)
    source_value = row.get("source_sample_id", sample_id.removesuffix(REPEATED_TARGET_ABLATION_SUFFIX))
    source_sample_id = sample_id.removesuffix(REPEATED_TARGET_ABLATION_SUFFIX) if pd.isna(source_value) else str(source_value)
    sequence, gt_mask, _sample = read_sequence_and_mask_for_sample(
        sample_id,
        source_sample_id=source_sample_id,
        input_condition=input_condition,
    )
    pred_mask = arrays["prediction_mask"].astype(np.uint8)
    frame_indices = arrays["frame_indices"].astype(int).tolist()
    condition_label = "target-repeat ablation" if input_condition == "target_repeat" else "normal sequence"
    sample_title = f"{row['sample_id']} | {condition_label} | {row['phase']} | Dice={row['dice']:.3f} | IoU={row['iou']:.3f}"

    frame_titles = [f"pos {i}\nfrm {frame_indices[i]}" for i in range(SEQUENCE_LENGTH)]
    make_contact_sheet(
        sequence,
        frame_titles,
        output_dir / "input_frames_contact_sheet.png",
        cmap="gray",
        highlight_indices={TARGET_IDX},
        figure_title=sample_title,
    )
    make_contact_sheet(
        arrays["encoder_normalized_cam"].astype(np.float32),
        frame_titles,
        output_dir / "encoder_heatmaps.png",
        cmap="inferno",
        highlight_indices={TARGET_IDX},
        figure_title=sample_title,
    )
    overlay_sheet(
        sequence,
        arrays["encoder_normalized_cam"].astype(np.float32),
        frame_titles,
        output_dir / "encoder_overlays_with_masks.png",
        target_position=TARGET_IDX,
        gt_mask=gt_mask,
        pred_mask=pred_mask,
        figure_title=sample_title,
    )

    combined_temporal_cams, combined_temporal_titles = combined_forward_backward_temporal_cams(arrays)
    make_contact_sheet(
        combined_temporal_cams,
        combined_temporal_titles,
        output_dir / "combined_forward_backward_temporal_heatmaps.png",
        cmap="inferno",
        highlight_indices={TARGET_IDX},
        figure_title=sample_title,
    )
    overlay_sheet(
        sequence,
        combined_temporal_cams,
        combined_temporal_titles,
        output_dir / "combined_forward_backward_temporal_overlays_with_masks.png",
        target_position=TARGET_IDX,
        gt_mask=gt_mask,
        pred_mask=pred_mask,
        figure_title=sample_title,
    )

    # Retain separate raw temporal heatmap sheets for branch-specific inspection, but keep overlays combined above.
    forward_titles = [f"F state {i}" for i in range(TARGET_IDX + 1)]
    forward_cams = arrays["forward_normalized_cam"].astype(np.float32)
    make_contact_sheet(forward_cams, forward_titles, output_dir / "forward_temporal_heatmaps.png", cmap="inferno", highlight_indices={TARGET_IDX}, figure_title=sample_title)
    overlay_sheet(
        sequence[:TARGET_IDX + 1],
        forward_cams,
        forward_titles,
        output_dir / "forward_temporal_overlays_with_masks.png",
        target_position=TARGET_IDX,
        gt_mask=gt_mask,
        pred_mask=pred_mask,
        figure_title=sample_title,
    )
    backward_positions = list(range(TARGET_IDX, SEQUENCE_LENGTH))
    backward_titles = [f"B chron pos {pos}" for pos in backward_positions]
    backward_cams = arrays["backward_normalized_cam"].astype(np.float32)
    make_contact_sheet(backward_cams, backward_titles, output_dir / "backward_temporal_heatmaps_chronological.png", cmap="inferno", highlight_indices={0}, figure_title=sample_title)
    overlay_sheet(
        sequence[TARGET_IDX:],
        backward_cams,
        backward_titles,
        output_dir / "backward_temporal_overlays_with_masks.png",
        target_position=0,
        gt_mask=gt_mask,
        pred_mask=pred_mask,
        figure_title=sample_title,
    )

    make_contact_sheet(pred_mask[None].astype(np.float32), ["Predicted mask"], output_dir / "predicted_mask.png", cmap="gray", figure_title=sample_title)
    make_contact_sheet(gt_mask[None], ["Ground-truth mask"], output_dir / "ground_truth_mask.png", cmap="gray", figure_title=sample_title)
    contour_overlay_figure(row, sequence, gt_mask, arrays, output_dir)

    plt.close("all")
    return {"sample_id": sample_id, "visualization_dir": str(output_dir.relative_to(OUTPUT_ROOT))}


In [ ]:
if not vis_df.empty:
    vis_rows = []
    for _, row in tqdm(vis_df.iterrows(), total=len(vis_df), desc="Writing visualization sets"):
        try:
            vis_rows.append(write_visualization_set(row))
        except Exception as exc:
            append_jsonl(FAILURE_MANIFEST, {"sample_id": str(row.get("sample_id", "unknown")), "visualization_error": repr(exc)})
            print(f"Visualization failed for {row.get('sample_id', 'unknown')}: {exc}")
        finally:
            gc.collect()
            plt.close("all")
    pd.DataFrame(vis_rows).to_csv(MANIFEST_DIR / "written_visualizations.csv", index=False)
    print(f"Visualization sets written: {len(vis_rows):,}")
else:
    print("No visualization subset available yet.")


## Aggregate Summary

This final cell summarizes processing, segmentation metrics, zero-heatmap counts, output disk usage, and visualization counts. It saves the same information to `summary.json`.


In [ ]:
def directory_size_bytes(path: Path) -> int:
    if not path.exists():
        return 0
    return sum(item.stat().st_size for item in path.rglob("*") if item.is_file())


def zero_heatmap_summary() -> dict[str, Any]:
    counts = {"encoder": {"zero": 0, "total": 0}, "forward": {"zero": 0, "total": 0}, "backward_recurrent": {"zero": 0, "total": 0}}
    for npz_path in ARRAY_DIR.glob("*.npz"):
        try:
            with np.load(npz_path, allow_pickle=False) as data:
                reps = data["zero_representation"].astype(str)
                zero = data["zero_after_normalization"].astype(bool)
                for rep in counts:
                    mask = reps == rep
                    counts[rep]["total"] += int(mask.sum())
                    counts[rep]["zero"] += int(zero[mask].sum())
        except Exception:
            continue
    for rep, values in counts.items():
        values["percent_zero"] = 100.0 * values["zero"] / values["total"] if values["total"] else 0.0
    return counts


def safe_read_csv(path: Path) -> pd.DataFrame:
    if not path.exists() or path.stat().st_size == 0:
        return pd.DataFrame()
    try:
        return pd.read_csv(path)
    except pd.errors.EmptyDataError:
        return pd.DataFrame()


metrics_df = read_jsonl(METRICS_JSONL)
if not metrics_df.empty:
    metrics_df = metrics_df.drop_duplicates(subset=["sample_id"], keep="last").reset_index(drop=True)
    metrics_df.to_csv(METRICS_CSV, index=False)
failures_df = read_jsonl(FAILURE_MANIFEST)
completion_df = read_jsonl(COMPLETION_MANIFEST)
visualization_written = safe_read_csv(MANIFEST_DIR / "written_visualizations.csv")

summary = {
    "run_mode": RUN_MODE,
    "checkpoint_path": str(CHECKPOINT_PATH),
    "checkpoint_id": CHECKPOINT_ID,
    "total_official_test_samples": len(all_test_samples),
    "active_samples": len(active_samples),
    "processed_metric_rows": int(len(metrics_df)),
    "completed_manifest_rows": int(len(completion_df)),
    "failed_rows": int(len(failures_df)),
    "dice_mean": float(metrics_df["dice"].mean()) if not metrics_df.empty else None,
    "dice_std": float(metrics_df["dice"].std()) if not metrics_df.empty else None,
    "iou_mean": float(metrics_df["iou"].mean()) if not metrics_df.empty else None,
    "iou_std": float(metrics_df["iou"].std()) if not metrics_df.empty else None,
    "segmentation_loss_mean": float(metrics_df["segmentation_loss"].mean()) if not metrics_df.empty else None,
    "phase_metrics": {},
    "zero_heatmaps": zero_heatmap_summary(),
    "output_disk_usage_bytes": directory_size_bytes(OUTPUT_ROOT),
    "visualization_samples_created": int(len(visualization_written)),
}

if not metrics_df.empty:
    for phase, group in metrics_df.groupby("phase"):
        summary["phase_metrics"][str(phase)] = {
            "count": int(len(group)),
            "dice_mean": float(group["dice"].mean()),
            "dice_std": float(group["dice"].std()),
            "iou_mean": float(group["iou"].mean()),
            "iou_std": float(group["iou"].std()),
            "segmentation_loss_mean": float(group["segmentation_loss"].mean()),
        }

with SUMMARY_PATH.open("w", encoding="utf-8") as file:
    json.dump(summary, file, indent=2)

print(json.dumps(summary, indent=2))
print(f"Summary saved to: {SUMMARY_PATH}")
print(f"Output folder: {OUTPUT_ROOT}")
print("Before ending the Kaggle session, download this folder or save it as a Kaggle Dataset.")
